# Example: Production Portfolio Engine

In this example, we build the full production loop — the bandit selects assets daily, the Cobb-Douglas allocator decides how much of each, sentiment monitoring adjusts lambda in real time, and escalation triggers provide the human safety net. We simulate 60 trading days on an HMM-generated path with a regime shift mid-period.

> **By the end of this example, you will be able to:**
> * Run the production loop (bandit + Cobb-Douglas) over a realistic simulated trading period
> * Implement sentiment monitoring with automatic lambda override when conditions deteriorate
> * Define and observe human escalation triggers firing during stress events

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

In [ ]:
let
    # Constants -
    global Δt = 1.0 / 252.0;
    global rf = 0.05;
    global B₀ = 10000.0;
    global N_short = 21;
    global N_long = 63;
    global N_growth = 10;
    global GAIN = 10.0;
    global offset = N_short + N_long;
    global n_production_days = 60;
    global T_total = offset + n_production_days + 20; # buffer

    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );
    global start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    println("Setup: $(n_production_days) production days, $(length(tickers)) tickers")
end

Generate a single HMM path with synthetic per-ticker prices for the production simulation.

In [ ]:
let
    # Fit a quick HMM for path generation -
    training = generate_training_prices(S₀=100.0, μ=0.08, σ=0.18, T=1260, seed=42);
    global hmm_model = hmm_fit(JumpHiddenMarkovModel, training; rf=rf, N=50, nu=5.0, dt=Δt);
    global hmm_model = hmm_tune(hmm_model, training;
        epsilon_range=range(1e-4, 2.5e-2, length=8),
        lambda_range=range(10.0, 80.0, length=6), n_paths=30);

    # Generate one path -
    result = hmm_simulate(hmm_model, T_total; n_paths=1);
    G = result.paths[1].observations;
    global market_prices = JumpHMM.prices_from_growth_rates(G, 100.0; rf=rf, dt=Δt);

    # Per-ticker prices via SIM -
    K = length(tickers);
    global price_matrix = zeros(T_total, K + 1);
    price_matrix[:, 1] = 1:T_total;
    gm_raw = compute_market_growth(market_prices; Δt=Δt);

    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
        price_matrix[1, k + 1] = start_prices[ticker];
        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
            price_matrix[t, k + 1] = price_matrix[t - 1, k + 1] * exp(gᵢ);
        end
    end

    # Compute EMAs, lambda, market growth -
    global ema_short = compute_ema(market_prices; window=N_short);
    global ema_long = compute_ema(market_prices; window=N_long);
    global lambda_series = compute_lambda(ema_short, ema_long; G=GAIN);
    lambda_series[1:offset] .= 0.0;
    global gm_ema = compute_ema(gm_raw; window=N_growth);

    # Generate synthetic sentiment -
    global sentiment_series = generate_synthetic_sentiment(market_prices; noise_σ=0.15, seed=77);

    println("Generated $(T_total)-day path with sentiment")
end

___
## Task 1: Build and Run the Production Loop
We define the production context (risk parameters, escalation thresholds) and run the full production simulation for 60 trading days. The bandit selects assets daily; the Cobb-Douglas allocator computes the optimal allocation; trigger rules enforce safety.

> **What should you see?** A wealth curve that responds to market conditions. The bandit may change its selected assets as conditions shift. The system should rebalance most days but may hold or de-risk when escalation triggers fire.

In [ ]:
let
    # Build production context -
    global prod_ctx = build(MyProductionContext, (
        tickers = tickers,
        sim_parameters = sim_params,
        B₀ = B₀,
        epsilon = 0.1,
        max_drawdown = 0.15,
        max_turnover = 0.50,
        sentiment_threshold = -0.5,
        sentiment_override_lambda = 2.0,  # very risk-averse when sentiment crashes
        max_bandit_churn = 2             # max 2 asset changes per day
    ));

    # Run production simulation -
    println("Running $(n_production_days)-day production simulation...")
    global (prod_results, prod_events) = run_production_simulation(
        prod_ctx, market_prices, price_matrix,
        sentiment_series, lambda_series, gm_ema;
        n_days=n_production_days, offset=offset, n_bandit_iters=100
    );

    println("Simulation complete:")
    println("  Final wealth: \$$(round(prod_results[end].wealth, digits=2))")
    println("  Escalation events: $(length(prod_events))")
    println("  Days rebalanced: $(sum(r.rebalanced for r in prod_results))")
end

**Visualize:** Production wealth curve with escalation markers.

In [ ]:
let
    days = [r.day for r in prod_results];
    wealth = [r.wealth for r in prod_results];

    p = plot(days, wealth, label="Production Wealth", linewidth=2.5, color=:steelblue,
        xlabel="Production Day", ylabel="Wealth (\$)",
        title="Production Portfolio Engine", size=(800, 400), legend=:topright)
    hline!(p, [B₀], label="Starting Capital", linestyle=:dash, color=:gray50, alpha=0.5)

    # Mark escalation events -
    for evt in prod_events
        if evt.day <= n_production_days
            color = evt.severity == :critical ? :red : :orange;
            vline!(p, [evt.day], label="", color=color, alpha=0.3, linewidth=2)
        end
    end

    # Add legend entries for escalation types -
    if any(e.severity == :critical for e in prod_events)
        plot!(p, [], [], label="Critical Escalation", color=:red, linewidth=2)
    end
    if any(e.severity == :warning for e in prod_events)
        plot!(p, [], [], label="Warning Escalation", color=:orange, linewidth=2)
    end
    p
end

___
## Task 2: Sentiment Monitoring with Lambda Override
We visualize how the sentiment signal interacts with the production system. When sentiment drops below the threshold ($s_t < -0.5$), the system overrides lambda to a high risk-aversion value, causing the allocator to shift toward safer assets.

> **What should you see?** The sentiment time series should correlate with market movements (positive during rallies, negative during drops). When it crosses the threshold, the effective lambda jumps up — and the portfolio response should be visible as reduced risk exposure.

In [ ]:
let
    days = [r.day for r in prod_results];
    sentiments = [r.sentiment for r in prod_results];
    lambdas = [r.lambda for r in prod_results];
    wealth = [r.wealth for r in prod_results];

    # Top: sentiment + threshold -
    p1 = plot(days, sentiments, label="Sentiment", linewidth=2, color=:purple)
    hline!(p1, [prod_ctx.sentiment_threshold], label="Threshold", linestyle=:dash, color=:red)
    hline!(p1, [0.0], label="", linestyle=:dot, color=:black, alpha=0.3)
    ylabel!(p1, "Sentiment Score")
    title!(p1, "Sentiment Monitoring")

    # Middle: effective lambda -
    p2 = plot(days, lambdas, label="Effective λ", linewidth=2, color=:coral)
    ylabel!(p2, "λ (risk-aversion)")
    title!(p2, "Lambda (with sentiment override)")

    # Bottom: wealth -
    p3 = plot(days, wealth, label="Wealth", linewidth=2, color=:steelblue)
    xlabel!(p3, "Production Day")
    ylabel!(p3, "Wealth (\$)")
    title!(p3, "Portfolio Response")

    plot(p1, p2, p3, layout=(3, 1), size=(800, 700), legend=:topright)
end

___
## Task 3: Human Override Rules and Escalation Log
We examine every escalation event that fired during the simulation — what triggered it, how severe it was, and what the system recommended. This is the audit trail that the investment committee reviews.

> **What should you see?** A table of escalation events with timestamps, trigger types, severity levels, and recommended actions. Critical events should correspond to large drawdowns; warnings to sentiment crashes or bandit churn.

In [ ]:
let
    if length(prod_events) > 0
        println("═"^70)
        println("  ESCALATION LOG: $(length(prod_events)) events")
        println("═"^70)

        escalation_df = DataFrame(
            "Day" => [e.day for e in prod_events],
            "Trigger" => [e.trigger_type for e in prod_events],
            "Severity" => [string(e.severity) for e in prod_events],
            "Message" => [e.message for e in prod_events],
            "Recommendation" => [e.recommended_action for e in prod_events]
        );
        pretty_table(escalation_df, tf=tf_markdown,
            columns_width=[5, 16, 10, 40, 40])
    else
        println("No escalation events during this simulation.")
        println("The market path was calm enough that no triggers fired.")
    end
end

**Visualize:** Bandit's selected asset subset over time — which assets did the bandit include each day?

In [ ]:
let
    K = length(tickers);
    days = [r.day for r in prod_results];
    n = length(days);

    # Build inclusion matrix -
    inclusion = zeros(n, K);
    for (i, r) in enumerate(prod_results)
        inclusion[i, :] = r.bandit_action;
    end

    colors = [:steelblue :coral :green :goldenrod :purple];
    p = plot(size=(800, 300), title="Bandit Asset Selection Over Time",
        xlabel="Production Day", ylabel="", yticks=(1:K, tickers), legend=false)

    for k in 1:K
        for i in 1:n
            if inclusion[i, k] == 1
                scatter!(p, [days[i]], [k], markersize=4, color=colors[k], alpha=0.7)
            end
        end
    end
    p
end

**Summary metrics.**

In [ ]:
let
    metrics = compute_dashboard_metrics(prod_results, prod_events);

    println("═"^50)
    println("  PRODUCTION SUMMARY")
    println("═"^50)
    println("  Days simulated:       $(metrics["n_days"])")
    println("  Final wealth:         \$$(round(metrics["final_wealth"], digits=2))")
    println("  Max drawdown:         $(round(metrics["max_drawdown"]*100, digits=1))%")
    println("  Rebalance days:       $(metrics["n_rebalances"])")
    println("  Bandit subset changes: $(metrics["n_bandit_changes"])")
    println("  Escalations:          $(metrics["n_escalations"]) ($(metrics["n_critical"]) critical, $(metrics["n_warning"]) warning)")
    println("  Avg sentiment:        $(round(metrics["avg_sentiment"], digits=3))")

    # Save for Example 2 -
    save_production_results(joinpath(_PATH_TO_DATA, "production-results.jld2"),
        prod_results, prod_events);
end

___
## Summary
In this example, we built and ran the full production portfolio engine:

- **Bandit + Cobb-Douglas** work together: the bandit selects which assets to include, the allocator decides how much of each
- **Sentiment monitoring** provides early warning — overriding lambda to increase risk-aversion before drawdowns materialize
- **Escalation triggers** fire when conditions breach thresholds, creating an audit trail for investment committee review
- Every decision is logged: shares, cash, bandit action, sentiment, lambda, rebalance flag, escalation flag

In the next example, we build an operational dashboard and stress-test the system with a crash injection.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.